In [ ]:
# ==============================================================
# HBN EEG PREPROCESSING — EPG-SD
# Preprocessing pipeline: ICA + avg re-reference + dead channel removal
# + window-level rejection + age/metadata saving
# Dataset: ds005516 (HBN Release 11, 430 participants)
# Execution environment: Python 3.11
# Reproducible preprocessing workflow for EPG-SD
# ==============================================================

# STEP 0: Install required dependencies when needed.
!pip install mne autoreject mne-icalabel openpyxl tqdm numpy -q

import os
import json
import numpy as np
import pandas as pd
import mne
from mne.preprocessing import ICA
from mne_icalabel import label_components   # auto-labels ICA components
from autoreject import get_rejection_threshold
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

# ==============================================================
# SECTION 1 — PATH CONFIGURATION
# ==============================================================
RAW_PATH = os.environ.get(
    "EPGSD_RAW_BIDS_DIR",
    "./data/ds005516",
)
SAVE_PATH = os.environ.get(
    "EPGSD_PREPROCESSED_DIR",
    "./data/preprocessed_HBN_v2",
)
PARTICIPANTS = os.environ.get(
    "EPGSD_PARTICIPANTS_FILE",
    "./data/participants.xlsx",
)

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(os.path.join(SAVE_PATH, "metadata"), exist_ok=True)

# ==============================================================
# SECTION 2 — PREPROCESSING PARAMETERS
# ==============================================================
SFREQ_TARGET   = 500          # HBN native sampling rate (Hz)
L_FREQ         = 1.0          # high-pass cut-off (Hz)
H_FREQ         = 40.0         # low-pass cut-off (Hz)
NOTCH_FREQ     = 60.0         # US power-line (HBN recorded in New York)
WIN_SAMPLES    = 500          # 1 second window at 500 Hz
STRIDE_SAMPLES = 250          # 50 % overlap
AMPLITUDE_THRESH = 4.0        # reject windows with any channel > 4 std
N_ICA_COMPONENTS = 20         # ICA components (sufficient for 128-ch after drops)
DEAD_CHAN_THRESH  = 1e-10      # channel is dead if std < this value

# Tasks available in HBN ds005516
VALID_TASKS = [
    "RestingState", "DespicableMe", "FunwithFractals",
    "ThePresent", "DiaryOfAWimpyKid", "contrastChangeDetection",
    "surroundSupp", "symbolSearch"
]

# ==============================================================
# SECTION 3 — LOAD PARTICIPANT METADATA (age, sex, scores)
# ==============================================================
def load_participants(xlsx_path):
    """Load participants.xlsx and return a dict keyed by participant_id."""
    df = pd.read_excel(xlsx_path, engine="openpyxl")
    df = df.dropna(subset=["participant_id"])
    df["participant_id"] = df["participant_id"].str.strip()
    meta = {}
    for _, row in df.iterrows():
        pid = row["participant_id"]
        meta[pid] = {
            "participant_id": pid,
            "age":            float(row["age"])        if pd.notna(row.get("age"))         else None,
            "sex":            str(row["sex"])           if pd.notna(row.get("sex"))          else None,
            "p_factor":       float(row["p_factor"])    if pd.notna(row.get("p_factor"))     else None,
            "attention":      float(row["attention"])   if pd.notna(row.get("attention"))    else None,
            "internalizing":  float(row["internalizing"]) if pd.notna(row.get("internalizing")) else None,
            "externalizing":  float(row["externalizing"]) if pd.notna(row.get("externalizing")) else None,
            "ehq_total":      float(row["ehq_total"])   if pd.notna(row.get("ehq_total"))    else None,
            # Task availability flags (available / caution / unavailable)
            "task_RestingState":           str(row.get("RestingState", "unavailable")),
            "task_DespicableMe":           str(row.get("DespicableMe", "unavailable")),
            "task_FunwithFractals":        str(row.get("FunwithFractals", "unavailable")),
            "task_ThePresent":             str(row.get("ThePresent", "unavailable")),
            "task_DiaryOfAWimpyKid":       str(row.get("DiaryOfAWimpyKid", "unavailable")),
        }
    print(f"Loaded metadata for {len(meta)} participants")
    return meta


# ==============================================================
# SECTION 4 — FILE DISCOVERY
# ==============================================================
def find_set_files(root):
    """Walk root and return all .set file paths sorted."""
    files = []
    for r, _, fs in os.walk(root):
        for f in fs:
            if f.endswith(".set"):
                files.append(os.path.join(r, f))
    return sorted(files)


def extract_subject_task(path):
    """Parse BIDS filename -> (subject_id, task_name)."""
    fname  = os.path.basename(path)
    parts  = fname.split("_")
    subject, task = "unknown", "unknown"
    for p in parts:
        if p.startswith("sub-"):
            subject = p
        if p.startswith("task-"):
            task = p.replace("task-", "").replace(".set", "")
    return subject, task


def build_existing_map(save_dir):
    """Return set of (subject, task) already preprocessed."""
    done = set()
    for f in os.listdir(save_dir):
        if f.endswith(".npy"):
            parts   = f.replace(".npy", "").split("_")
            subject = next((p for p in parts if p.startswith("sub-")), None)
            task    = next((p.replace("task-", "") for p in parts if p.startswith("task-")), None)
            if subject and task:
                done.add((subject, task))
    return done


# ==============================================================
# SECTION 5 — CORE PREPROCESSING FUNCTION
# ==============================================================
def preprocess_one(raw_path, subject, task, participant_meta):
    """
    Full preprocessing pipeline for one .set file.
    Returns: windows (np.ndarray, shape [N, 128, 500]), metadata dict
    Raises ValueError when a recording does not meet quality-control criteria.
    """

    # --- 5.1  Load raw EEG -------------------------------------------
    raw = mne.io.read_raw_eeglab(raw_path, preload=True, verbose=False)

    # Enforce the 500-Hz sampling rate reported in the manuscript.
    if not np.isclose(float(raw.info["sfreq"]), SFREQ_TARGET):
        raw.resample(SFREQ_TARGET, npad="auto", verbose=False)

    # --- 5.2  Keep only EEG channels ----------------------------------
    raw.pick_types(eeg=True, exclude="bads")

    # --- 5.3  Remove dead / flat channels -----------------------------
    data_all = raw.get_data()
    ch_stds   = data_all.std(axis=1)
    dead_idx  = np.where(ch_stds < DEAD_CHAN_THRESH)[0]
    dead_names = [raw.ch_names[i] for i in dead_idx]
    if dead_names:
        raw.drop_channels(dead_names)
        print(f"   WARNING: Dropped {len(dead_names)} dead channel(s): {dead_names}")

    n_channels = len(raw.ch_names)
    if n_channels < 64:
        raise ValueError(f"Too few channels after dead-channel removal: {n_channels}")

    # --- 5.4  Average re-reference ------------------------------------
    raw.set_eeg_reference("average", projection=False, verbose=False)

    # --- 5.5  Band-pass filter ----------------------------------------
    raw.filter(L_FREQ, H_FREQ, method="fir", fir_window="hamming", verbose=False)

    # --- 5.6  Notch filter (60 Hz US power-line) ----------------------
    raw.notch_filter(NOTCH_FREQ, verbose=False)

    # --- 5.7  ICA artifact removal ------------------------------------
    n_ica = min(N_ICA_COMPONENTS, n_channels - 1)
    ica = ICA(
        n_components=n_ica,
        method="fastica",
        max_iter=400,
        random_state=42,
        verbose=False
    )
    ica.fit(raw, verbose=False)

    # Auto-label components: removes eye blinks, muscle, heartbeat
    try:
        ic_labels = label_components(raw, ica, method="iclabel")
        exclude_labels = ["eye blink", "muscle artifact", "heart beat", "line noise"]
        bad_ics = [
            idx for idx, (lbl, prob) in enumerate(
                zip(ic_labels["labels"], ic_labels["y_pred_proba"])
            )
            if lbl in exclude_labels and prob > 0.70
        ]
        ica.exclude = bad_ics
        print(f"   ICA excluded {len(bad_ics)} artifact component(s)")
    except Exception as e:
        raise RuntimeError(
            f"ICLabel failed for {raw_path}; preprocessing cannot continue: {e}"
        ) from e

    ica.apply(raw, verbose=False)

    # --- 5.8  Get cleaned data ----------------------------------------
    data = raw.get_data()           # shape: (n_channels, n_times)
    n_ch, n_times = data.shape

    # --- 5.9  Global z-score per channel (across full recording) ------
    ch_mean = data.mean(axis=1, keepdims=True)
    ch_std  = data.std(axis=1,  keepdims=True) + 1e-8
    data    = (data - ch_mean) / ch_std

    # --- 5.10  Sliding-window segmentation ----------------------------
    windows = []
    for start in range(0, n_times - WIN_SAMPLES + 1, STRIDE_SAMPLES):
        w = data[:, start : start + WIN_SAMPLES]    # (n_ch, 500)
        # Window-level artifact rejection: drop if any value > AMPLITUDE_THRESH std
        if np.abs(w).max() > AMPLITUDE_THRESH:
            continue
        windows.append(w)

    if len(windows) < 10:
        raise ValueError(f"Only {len(windows)} clean windows — recording too short or too noisy")

    windows = np.array(windows, dtype=np.float32)   # (N, n_ch, 500)

    # --- 5.11  Pad/trim to exactly 128 channels for model consistency -
    # (HBN 10-20 layout has 128 EEG + 1 reference = 129 raw; after drop -> 128)
    if windows.shape[1] > 128:
        windows = windows[:, :128, :]
    elif windows.shape[1] < 128:
        pad = np.zeros((windows.shape[0], 128 - windows.shape[1], WIN_SAMPLES), dtype=np.float32)
        windows = np.concatenate([windows, pad], axis=1)

    # --- 5.12  Build per-file metadata --------------------------------
    meta = {
        "subject":         subject,
        "task":            task,
        "n_windows":       int(windows.shape[0]),
        "n_channels":      int(windows.shape[1]),
        "win_samples":     WIN_SAMPLES,
        "stride_samples":  STRIDE_SAMPLES,
        "sfreq":           SFREQ_TARGET,
        "ica_excluded":    len(ica.exclude),
        "dead_channels":   dead_names,
        # Participant-level info from xlsx
        "age":             participant_meta.get("age"),
        "sex":             participant_meta.get("sex"),
        "p_factor":        participant_meta.get("p_factor"),
        "attention":       participant_meta.get("attention"),
        "internalizing":   participant_meta.get("internalizing"),
        "externalizing":   participant_meta.get("externalizing"),
        "ehq_total":       participant_meta.get("ehq_total"),
    }

    return windows, meta


# ==============================================================
# SECTION 6 — MAIN LOOP
# ==============================================================
def run_preprocessing():
    print("\n" + "="*60)
    print("  HBN EEG PREPROCESSING  —  EPG-SD Framework")
    print("="*60)

    # Load participant metadata from xlsx
    participant_meta = load_participants(PARTICIPANTS)

    # Discover all raw .set files
    raw_files    = find_set_files(RAW_PATH)
    existing_map = build_existing_map(SAVE_PATH)

    print(f"\nTotal .set files found : {len(raw_files)}")
    print(f"Already preprocessed   : {len(existing_map)}")
    print(f"Remaining              : {len(raw_files) - len(existing_map)}\n")

    processed_count = 0
    skipped_count   = 0
    error_count     = 0
    error_log       = []

    for fpath in tqdm(raw_files, desc="Preprocessing"):
        subject, task = extract_subject_task(fpath)

        # Skip previously processed recordings.
        if (subject, task) in existing_map:
            skipped_count += 1
            continue

        # Skip tasks outside the configured set.
        if task not in VALID_TASKS:
            skipped_count += 1
            continue

        tqdm.write(f"\n{'─'*50}")
        tqdm.write(f"Subject : {subject}")
        tqdm.write(f"Task    : {task}")

        # Retrieve participant metadata.
        p_meta = participant_meta.get(subject, {})
        age    = p_meta.get("age", "N/A")
        sex    = p_meta.get("sex", "N/A")
        tqdm.write(f"Age     : {age}  |  Sex: {sex}")

        try:
            windows, meta = preprocess_one(fpath, subject, task, p_meta)

            # Save accepted windows.
            npy_name  = f"{subject}_task-{task}.npy"
            npy_path  = os.path.join(SAVE_PATH, npy_name)
            np.save(npy_path, windows)

            # Save recording metadata.
            json_name = f"{subject}_task-{task}.json"
            json_path = os.path.join(SAVE_PATH, "metadata", json_name)
            with open(json_path, "w") as jf:
                json.dump(meta, jf, indent=2)

            tqdm.write(f"Saved {windows.shape[0]} windows -> {npy_name}")
            tqdm.write(f"   Age={age}, Sex={sex}, ICA_excluded={meta['ica_excluded']}")

            existing_map.add((subject, task))
            processed_count += 1

        except Exception as e:
            error_msg = f"ERROR: ERROR: {subject} | {task} -> {str(e)}"
            tqdm.write(error_msg)
            error_log.append({"subject": subject, "task": task, "error": str(e)})
            error_count += 1
            continue

    # Processing summary.────
    print("\n" + "="*60)
    print("  PREPROCESSING COMPLETE")
    print("="*60)
    print(f"  Newly processed : {processed_count}")
    print(f"  Skipped        : {skipped_count}")
    print(f"  ERROR: Errors          : {error_count}")

    if error_log:
        err_path = os.path.join(SAVE_PATH, "error_log.json")
        with open(err_path, "w") as f:
            json.dump(error_log, f, indent=2)
        print(f"\n  Error log saved -> {err_path}")

    # Build the master metadata table.
    build_master_csv(SAVE_PATH)


# ==============================================================
# SECTION 7 — MASTER METADATA CSV
# (one row per subject-task, with age/sex/scores/n_windows)
# ==============================================================
def build_master_csv(save_dir):
    """Collect all per-file JSON metadata into one CSV for easy label loading."""
    meta_dir = os.path.join(save_dir, "metadata")
    rows = []
    for fname in sorted(os.listdir(meta_dir)):
        if fname.endswith(".json"):
            with open(os.path.join(meta_dir, fname)) as f:
                rows.append(json.load(f))

    if not rows:
        print("No metadata files found yet.")
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, "master_metadata.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nMaster metadata CSV saved -> {csv_path}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Rows   : {len(df)}")
    print(df[["subject", "task", "age", "sex", "n_windows", "ica_excluded"]].head(10).to_string(index=False))


# ==============================================================
# SECTION 8 — QUICK VERIFICATION (run after preprocessing)
# ==============================================================
def verify_preprocessed_file(npy_path, json_path=None):
    """
    Load one preprocessed file and print a quality report.
    Usage: verify_preprocessed_file('/path/to/sub-XXX_task-YYY.npy')
    """
    data = np.load(npy_path)
    print(f"\n{'='*50}")
    print(f"File   : {os.path.basename(npy_path)}")
    print(f"Shape  : {data.shape}  (windows, channels, samples)")
    print(f"Dtype  : {data.dtype}")
    print(f"Min    : {data.min():.4f}")
    print(f"Max    : {data.max():.4f}")
    print(f"Mean   : {data.mean():.6f}  (should be ~0)")
    print(f"Std    : {data.std():.4f}  (should be ~1)")
    print(f"NaN    : {np.isnan(data).sum()}")
    print(f"Inf    : {np.isinf(data).sum()}")

    # Dead channels check
    flat_ch = (data.std(axis=-1) < 1e-6).sum()
    print(f"Flat channels (any window): {flat_ch}  (should be 0)")

    # Artifact check
    over_thresh = (np.abs(data) > 4.0).sum()
    pct = 100 * over_thresh / data.size
    print(f"Samples >4 std: {over_thresh} ({pct:.4f}%)  (should be near 0)")

    if json_path and os.path.exists(json_path):
        with open(json_path) as f:
            meta = json.load(f)
        print(f"\nMetadata:")
        for k, v in meta.items():
            print(f"  {k:20s}: {v}")


# ==============================================================
# ENTRY POINT
# ==============================================================
if __name__ == "__main__":
    run_preprocessing()

    # Optional verification example:
    # verify_preprocessed_file(
    #     "./data/preprocessed_HBN_v2/example_recording.npy",
    #     "./data/preprocessed_HBN_v2/metadata/example_recording.json",
    # )